# Main 8: HydrAI — GRU axial PFR profile evolution (smoke / pipeline validation)

Sequence experiment: each Cantera run is an ordered profile along the reactor. A **ProfileGRU** predicts **scaled state increments** with teacher forcing (train) and autoregressive rollout (val/test).

**Intent:** Use the **production pkl**. In **Section 2**, set `SUBSAMPLE_RUNS` / `SUBSAMPLE_MAX_RUNS` for a fast, stratified dev subset (same design coverage as full data). Set `SUBSAMPLE_RUNS = False` for full training (~45k runs). Increase capacity via `sequence_rnn_production` when scaling up.

**Prerequisites:** Main_2, Main_3 (`features_targets_*.pkl` with lumped species).

**Sections:** 1 Setup · 2 Paths · 3 Data · 4 Columns & τ · 5 Split & sequences · 6 Model · 7 Train · 8 Evaluate · 9 Plots · 10 Export


In [ ]:
# 1. SETUP
import os, sys, glob, json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle
from src.utils.residence_time import add_residence_time_columns, validate_residence_time_on_df
from src.ml.sequence_dataset import (
    SequenceColumnSpec,
    build_sequence_arrays,
    ProfileSequenceDataset,
    collate_profile_sequences,
    masked_delta_mse,
    masked_rollout_mse,
    run_id_series,
)
from src.ml.profile_gru import ProfileGRU
from src.ml.profile_rollout_eval import eval_rollout_loss
from src.ml.profile_rollout_stability import (
    clamp_state_physical,
    envelope_sanity_summary,
    first_divergence_report,
    first_nonphysical_divergence_report,
    plot_y_limits_from_true,
    rollout_minmax_table,
)
from src.ml.sequence_rnn_dev import (
    assess_run_level_scale,
    metrics_disclaimer,
    print_per_variable_metrics,
    print_safeguards_checklist,
    print_scale_report,
    rollout_divergence_summary,
    subset_sequence_dict,
    subsample_runs_representative,
)
from src.utils.profile_predictions import anchor_inlet_profile_predictions
from src.utils.training_progress_log import (
    MAIN_8_STEM,
    init_training_progress_log,
    append_training_progress,
    training_progress_log_path,
)

setup_matplotlib()
start_run_log("Main_8_train_evaluate_GRU_profile_evolution")

# Re-run this cell after editing src/ml/*.py (avoids stale imports without full kernel restart)
import importlib
import src.ml.sequence_dataset as _ml_sd
import src.ml.profile_gru as _ml_pg
import src.ml.profile_rollout_eval as _ml_pre
import src.ml.profile_rollout_stability as _ml_prs
import src.ml.sequence_rnn_dev as _ml_srd
for _m in (_ml_sd, _ml_pg, _ml_pre, _ml_prs, _ml_srd):
    importlib.reload(_m)
masked_delta_mse = _ml_sd.masked_delta_mse
masked_rollout_mse = _ml_sd.masked_rollout_mse
eval_rollout_loss = _ml_pre.eval_rollout_loss
print_per_variable_metrics = _ml_srd.print_per_variable_metrics
envelope_sanity_summary = _ml_prs.envelope_sanity_summary
rollout_minmax_table = _ml_prs.rollout_minmax_table

print("Setup OK.")


In [ ]:
# 1b. DEVICE
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"PyTorch {torch.__version__} | device={device}")


In [ ]:
# 2. PATHS & FLAGS
IF_PLOT_SHOWN = True
IF_PLOT_EXPORT = True
IF_MODEL_EXPORT = True
WRITE_TRAINING_PROGRESS_LOG = True

# --- Data size (production pkl; stratified by design + axial length) ---
SUBSAMPLE_RUNS = True              # False = all Cantera runs
SUBSAMPLE_MAX_RUNS = 400           # used only when SUBSAMPLE_RUNS is True
SUBSAMPLE_STRATA = 4
SUBSAMPLE_BY_AXIAL_LENGTH = True
FAST_TRAINING = True 
N_CPU_CORES = 10

from src.utils.cpu_threads import configure_cpu_threads
_CPU = configure_cpu_threads(N_CPU_CORES, 1, device.type)

CONFIG_PATH = project_root / "configs" / "ml" / "ml_training_config.json"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
EXPORT_DIR = project_root / "models"
FIG_DIR = project_root / "outputs" / "figures" / MAIN_8_STEM
TRAINING_PROGRESS_LOG = training_progress_log_path(project_root, MAIN_8_STEM)

print(f"CPU cores={_CPU['n_cores']} | plot export={IF_PLOT_EXPORT}")


In [ ]:
# 3. CONFIG & DATA
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
else:
    config = {}

TEST_SIZE = config.get("test_size", 0.2)
RANDOM_STATE = config.get("random_state", 42)
RNN_CFG = config.get("sequence_rnn", {})
HIDDEN_SIZE = int(RNN_CFG.get("hidden_size", 64))
DESIGN_HIDDEN = int(RNN_CFG.get("design_hidden", 32))
EARLY_STOPPING_METRIC = RNN_CFG.get("early_stopping_metric", "validation_rollout_mse")
NUM_GRU_LAYERS = int(RNN_CFG.get("num_gru_layers", 1))
DROPOUT = float(RNN_CFG.get("dropout", 0.0))
EPOCHS = int(RNN_CFG.get("epochs", 50))
BATCH_SIZE = int(RNN_CFG.get("batch_size", 8))
LEARNING_RATE = float(RNN_CFG.get("learning_rate", 3e-4))
VAL_FRACTION = float(RNN_CFG.get("val_fraction", 0.15))
GRAD_CLIP_NORM = float(RNN_CFG.get("grad_clip_norm", 1.0))
MAX_DELTA_SCALED = RNN_CFG.get("max_delta_scaled", 2.0)
MAX_DELTA_SCALED = float(MAX_DELTA_SCALED) if MAX_DELTA_SCALED is not None else None
MAX_STATE_SCALED = float(RNN_CFG.get("max_state_scaled", 10.0))
_vrmax = RNN_CFG.get("val_rollout_max_steps")
VAL_ROLLOUT_MAX_STEPS = int(_vrmax) if _vrmax is not None else None
SCHED_SAMPLING_MAX = float(RNN_CFG.get("scheduled_sampling_max_prob", 0.35))
SCHED_SAMPLING_RAMP = int(RNN_CFG.get("scheduled_sampling_ramp_epochs", 25))
ROLLOUT_LOSS_WEIGHT = float(RNN_CFG.get("rollout_loss_weight", 0.25))
VAL_EVAL_EVERY = int(RNN_CFG.get("val_eval_every", 1))
_vmax = RNN_CFG.get("val_max_runs")
VAL_MAX_RUNS = int(_vmax) if _vmax is not None else None
TRAIN_ROLLOUT_AFTER_EPOCH = int(RNN_CFG.get("train_rollout_after_epoch", 1))
if not FAST_TRAINING:
    VAL_EVAL_EVERY = 1
    VAL_MAX_RUNS = None
    TRAIN_ROLLOUT_AFTER_EPOCH = 1
    if _vrmax is None:
        VAL_ROLLOUT_MAX_STEPS = None
APPLY_PHYSICAL_CLAMP_EVAL = bool(RNN_CFG.get("apply_physical_clamp_eval", True))
MIN_RUNS_STABLE = int(RNN_CFG.get("min_runs_for_stable_metrics", 20))
MIN_RUNS_PER_SPLIT = int(RNN_CFG.get("min_runs_per_split", 3))

DEV_MAX_RUNS = int(SUBSAMPLE_MAX_RUNS) if SUBSAMPLE_RUNS else None
DEV_SUBSAMPLE_STRATA = int(SUBSAMPLE_STRATA)
DEV_SUBSAMPLE_AXIAL = bool(SUBSAMPLE_BY_AXIAL_LENGTH)
if SUBSAMPLE_RUNS:
    print(f"Data size: SUBSAMPLE_RUNS=True, max_runs={SUBSAMPLE_MAX_RUNS:,}")
else:
    print("Data size: SUBSAMPLE_RUNS=False (full production set)")

print(
    f"GRU: hidden={HIDDEN_SIZE}, design_hidden={DESIGN_HIDDEN}, layers={NUM_GRU_LAYERS}, "
    f"dropout={DROPOUT}, lr={LEARNING_RATE}, grad_clip={GRAD_CLIP_NORM}, "
    f"batch_size={BATCH_SIZE}, epochs={EPOCHS}, max_delta_scaled={MAX_DELTA_SCALED}, "
    f"max_state_scaled={MAX_STATE_SCALED}"
)
print_safeguards_checklist(
    grad_clip_norm=GRAD_CLIP_NORM,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    batch_size=BATCH_SIZE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
)

pkl_files = sorted(glob.glob(str(PROCESSED_DATA_DIR / "features_targets_*.pkl")), reverse=True)
if not pkl_files:
    raise FileNotFoundError(f"No features_targets_*.pkl in {PROCESSED_DATA_DIR}. Run Main_3.")

DATA_FILE = pkl_files[0]
_m7_manifest_path = EXPORT_DIR / "simple_nn_full_profile_manifest.json"
if _m7_manifest_path.exists():
    with open(_m7_manifest_path) as _f:
        _m7_meta = json.load(_f)
    _m7_pkl = _m7_meta.get("data_file")
    if _m7_pkl:
        _cand = PROCESSED_DATA_DIR / Path(_m7_pkl).name
        if _cand.exists():
            DATA_FILE = str(_cand)
            print(f"[INFO] Using Main_7 manifest data_file: {_cand.name}")
        else:
            print(f"[WARN] Main_7 data_file {_m7_pkl!r} not found; using latest pkl.")
loaded = load_portable_pickle(DATA_FILE)
df_features = loaded["df_features"]
df_target = loaded["df_target"]
print(f"Data: {Path(DATA_FILE).name} | rows={len(df_features):,}")


In [ ]:
# 4. COLUMNS, MERGE, RESIDENCE TIME
design_cols = [
    "initial_temperature_K",
    "initial_pressure_Pa",
    "reactor_length_m",
    "reactor_diameter_m",
    "mass_flow_rate_kgps",
    "heat_flux_Wm2",
]
if "reactant_type" in df_features.columns:
    design_cols.append("reactant_type")

run_cols = [c for c in design_cols if c in df_features.columns]

_lump_cols = [c for c in df_target.columns if c.startswith("Y_lump_")]
species_cols = sorted(_lump_cols) if _lump_cols else sorted(
    c for c in df_target.columns if c.startswith("Y_")
)

state_cols = [c for c in [
    "temperature_K", "pressure_Pa", "density_kgm3", "velocity_ms",
] if c in df_target.columns] + species_cols

merge_cols = ["z_position_m", "relative_position"] + [
    c for c in design_cols if c in df_features.columns
]
df = df_features[merge_cols].join(df_target[state_cols], how="inner")

if "velocity_ms" not in df.columns:
    raise KeyError("velocity_ms required for residence time and state vector.")

label_encoder = None
if "reactant_type" in df.columns:
    label_encoder = LabelEncoder()
    df["reactant_type"] = label_encoder.fit_transform(df["reactant_type"].astype(str))

df = add_residence_time_columns(df, run_cols=run_cols, on_error="warn")
full_run_id = run_id_series(df, run_cols)

if DEV_MAX_RUNS is not None:
    _n_before = int(full_run_id.nunique())
    _kept = subsample_runs_representative(
        df,
        full_run_id,
        run_cols,
        DEV_MAX_RUNS,
        random_state=RANDOM_STATE,
        n_strata=DEV_SUBSAMPLE_STRATA,
        include_profile_length=DEV_SUBSAMPLE_AXIAL,
    )
    _mask = full_run_id.isin(_kept).to_numpy()
    df = df.loc[_mask].copy()
    full_run_id = run_id_series(df, run_cols)
    print(
        f"[DEV] Stratified subsample: {len(_kept):,} / {_n_before:,} runs, "
        f"{len(df):,} rows (set SUBSAMPLE_RUNS=False in §2 for full data)"
    )

_rt_errors = validate_residence_time_on_df(df, run_cols=run_cols)
if _rt_errors:
    raise ValueError(
        "Residence time validation failed for "
        f"{len(_rt_errors)} run(s). First: {_rt_errors[0]}"
    )
print(
    f"Residence time OK on {full_run_id.nunique():,} runs "
    f"(tau_total_s range {df['tau_total_s'].min():.4g}-{df['tau_total_s'].max():.4g} s)"
)

exog_cols = ["relative_position", "relative_tau", "log_dt_s"]
spec = SequenceColumnSpec(
    run_cols=run_cols,
    design_cols=[c for c in design_cols if c in df.columns],
    exog_cols=exog_cols,
    state_cols=state_cols,
)
N_RUNS_TOTAL = int(full_run_id.nunique())
print(f"Runs={N_RUNS_TOTAL:,} | state_dim={len(state_cols)} | exog={exog_cols}")
if N_RUNS_TOTAL < MIN_RUNS_STABLE:
    _prod = config.get("sequence_rnn_production", {})
    if _prod:
        print(
            "[SMOKE] For production-scale training later, consider config "
            f"sequence_rnn_production: hidden_size={_prod.get('hidden_size', '?')}, "
            f"epochs={_prod.get('epochs', '?')}"
        )


In [ ]:
# 5. RUN-LEVEL SPLIT, SCALERS, SEQUENCE ARRAYS (by run id — never by axial row)
unique_runs = np.array(sorted(pd.unique(full_run_id)))
assert len(unique_runs) == len(np.unique(full_run_id)), "run ids must be one per Cantera run"
trainval_runs, test_runs = train_test_split(
    unique_runs, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
train_runs, val_runs = train_test_split(
    trainval_runs, test_size=VAL_FRACTION, random_state=RANDOM_STATE,
)


def mask_for_runs(run_ids):
    return full_run_id.isin(run_ids).to_numpy()


train_mask = mask_for_runs(train_runs)
val_mask = mask_for_runs(val_runs)
test_mask = mask_for_runs(test_runs)

scaler_design = StandardScaler()
design_train = df.loc[train_mask].groupby(run_cols, dropna=False)[spec.design_cols].first()
scaler_design.fit(design_train)

scaler_state = StandardScaler()
scaler_state.fit(df.loc[train_mask, state_cols])

scaler_exog = StandardScaler()
scaler_exog.fit(df.loc[train_mask, exog_cols])


def scaled_sequences(run_mask):
    arrays = build_sequence_arrays(
        df.loc[run_mask], spec, full_run_id[run_mask].to_numpy(),
    )
    for seq in arrays.values():
        seq["design"] = scaler_design.transform(seq["design"].reshape(1, -1)).ravel()
        seq["state"] = scaler_state.transform(seq["state"])
        seq["exog"] = scaler_exog.transform(seq["exog"])
    return arrays


seq_train = scaled_sequences(train_mask)
seq_val = scaled_sequences(val_mask)
seq_test = scaled_sequences(test_mask)
seq_val_eval = subset_sequence_dict(seq_val, VAL_MAX_RUNS, seed=RANDOM_STATE)

train_loader = DataLoader(
    ProfileSequenceDataset(seq_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_profile_sequences,
)
val_loader = DataLoader(
    ProfileSequenceDataset(seq_val_eval),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_profile_sequences,
)
if len(seq_val_eval) < len(seq_val):
    print(
        f"Val rollout metric: {len(seq_val_eval)} / {len(seq_val)} runs "
        f"(VAL_MAX_RUNS={VAL_MAX_RUNS}); every {VAL_EVAL_EVERY} epoch(s)"
    )
test_loader = DataLoader(
    ProfileSequenceDataset(seq_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_profile_sequences,
)

_profile_lengths = df.groupby(run_cols, dropna=False).size()
if _profile_lengths.nunique() > 1:
    print(f"[WARN] Unequal axial lengths per run: {_profile_lengths.value_counts().head().to_dict()}")
else:
    print(f"Axial points per run (constant): {int(_profile_lengths.iloc[0])}")

assert not (train_mask & test_mask).any(), "Train/test row leakage"
assert not (val_mask & test_mask).any(), "Val/test row leakage"
print(f"Train/val/test runs: {len(seq_train)} / {len(seq_val)} / {len(seq_test)}")

SCALE_REPORT = assess_run_level_scale(
    N_RUNS_TOTAL,
    len(seq_train),
    len(seq_val),
    len(seq_test),
    min_runs_stable_metrics=MIN_RUNS_STABLE,
    min_runs_per_split=MIN_RUNS_PER_SPLIT,
)
SMOKE_MODE = SCALE_REPORT.smoke_mode
print_scale_report(SCALE_REPORT)


In [ ]:
# 6. PROFILE GRU
torch.manual_seed(RANDOM_STATE)
model = ProfileGRU(
    n_state=len(state_cols),
    n_exog=len(exog_cols),
    n_design=len(spec.design_cols),
    hidden_size=HIDDEN_SIZE,
    design_hidden=DESIGN_HIDDEN,
    num_gru_layers=NUM_GRU_LAYERS,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable parameters: {n_params:,}")


In [ ]:
# 7. TRAINING (teacher forcing + scheduled sampling; val = finite rollout MSE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

if WRITE_TRAINING_PROGRESS_LOG:
    init_training_progress_log(TRAINING_PROGRESS_LOG)
    print(f"Training progress log: {TRAINING_PROGRESS_LOG}")

LOG_EVERY = max(1, EPOCHS // 20)
print(
    f"Stability: max_delta_scaled={MAX_DELTA_SCALED}, max_state_scaled={MAX_STATE_SCALED}, "
    f"scheduled_sampling→{SCHED_SAMPLING_MAX} over {SCHED_SAMPLING_RAMP} epochs, "
    f"rollout_loss_weight={ROLLOUT_LOSS_WEIGHT}"
)
print(
    f"Speed: FAST_TRAINING={FAST_TRAINING}, val_every={VAL_EVAL_EVERY}, "
    f"val_rollout_max_steps={VAL_ROLLOUT_MAX_STEPS}, train_rollout_after_epoch={TRAIN_ROLLOUT_AFTER_EPOCH}, "
    f"batch_size={BATCH_SIZE}, device={device}"
)
if device.type == "cpu":
    print("[TIP] GPU/MPS greatly speeds rollout; val rollout is the main cost per epoch.")

best_val = float("inf")
best_state = None
first_val_bad = None

for epoch in range(1, EPOCHS + 1):
    ss_prob = SCHED_SAMPLING_MAX * min(1.0, epoch / max(1, SCHED_SAMPLING_RAMP))
    model.train()
    train_losses = []
    for batch in train_loader:
        st = batch["state"].to(device)
        ex = batch["exog"].to(device)
        des = batch["design"].to(device)
        lengths = batch["lengths"].to(device)
        pred_delta = model.forward_teacher_forcing(st, ex, des, lengths=lengths)
        loss_tf = masked_delta_mse(pred_delta, st, lengths.cpu())
        loss = loss_tf
        use_rollout_loss = (
            ROLLOUT_LOSS_WEIGHT > 0 and ss_prob > 0 and epoch >= TRAIN_ROLLOUT_AFTER_EPOCH
        )
        if use_rollout_loss:
            pred_ro = model.forward_rollout(
                st[:, 0, :], ex, des, lengths=lengths,
                max_delta_scaled=MAX_DELTA_SCALED,
                max_state_scaled=MAX_STATE_SCALED,
                true_states=st,
                scheduled_sampling_prob=ss_prob,
            )
            loss_ro = masked_rollout_mse(pred_ro, st, lengths.cpu(), reduction="mean")
            if torch.isfinite(loss_ro).item():
                loss = (1.0 - ROLLOUT_LOSS_WEIGHT) * loss_tf + ROLLOUT_LOSS_WEIGHT * loss_ro
        optimizer.zero_grad()
        loss.backward()
        if GRAD_CLIP_NORM and GRAD_CLIP_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        train_losses.append(loss_tf.item())

    train_mse = float(np.mean(train_losses))
    do_val_rollout = (VAL_EVAL_EVERY <= 1) or (epoch % VAL_EVAL_EVERY == 0) or epoch == EPOCHS
    if do_val_rollout:
        val_mse, bad = eval_rollout_loss(
            model, val_loader, device,
            max_delta_scaled=MAX_DELTA_SCALED,
            max_state_scaled=MAX_STATE_SCALED,
            max_rollout_steps=VAL_ROLLOUT_MAX_STEPS,
            state_cols=state_cols,
            log_first_bad=(epoch == 1 or first_val_bad is None),
        )
        if bad is not None and first_val_bad is None:
            first_val_bad = bad
        if np.isfinite(val_mse) and val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        val_mse = float("nan")

    if WRITE_TRAINING_PROGRESS_LOG:
        append_training_progress(
            TRAINING_PROGRESS_LOG,
            epoch,
            train_mse,
            test_mse=val_mse if do_val_rollout and np.isfinite(val_mse) else None,
            lr=LEARNING_RATE,
            is_checkpoint=do_val_rollout and np.isfinite(val_mse),
        )

    if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
        val_rmse = float(np.sqrt(val_mse)) if np.isfinite(val_mse) else float("nan")
        val_tag = "rollout" if do_val_rollout else "skipped"
        print(
            f"Epoch {epoch:4d} | train TF delta-MSE={train_mse:.6f} | "
            f"val {val_tag}-MSE={val_mse:.6f} (RMSE={val_rmse:.6f}) | ss_p={ss_prob:.2f}"
        )

if best_state is not None:
    model.load_state_dict(best_state)
    print(
        f"Checkpoint: best {EARLY_STOPPING_METRIC}={best_val:.6f} "
        f"(rollout RMSE={np.sqrt(best_val):.6f} scaled)"
    )
else:
    print("[WARN] No finite val rollout checkpoint — using last-epoch weights.")
    if first_val_bad is not None:
        print(f"  First val non-finite: {first_val_bad}")


In [ ]:
# 8. EVALUATION - teacher-forced vs rollout (physical units)
# Rollout loop: scaled state + scaled delta only. inverse_transform / clamps apply here for metrics.
import importlib
import src.ml.sequence_rnn_dev as _ml_srd
import src.ml.profile_rollout_stability as _ml_prs
importlib.reload(_ml_srd)
importlib.reload(_ml_prs)
print_per_variable_metrics = _ml_srd.print_per_variable_metrics
envelope_sanity_summary = _ml_prs.envelope_sanity_summary
rollout_minmax_table = _ml_prs.rollout_minmax_table

@torch.no_grad()
def collect_predictions(loader, mode="rollout"):
    model.eval()
    rows = []
    for batch in loader:
        st = batch["state"].cpu().numpy()
        ex = batch["exog"].cpu().numpy()
        des = batch["design"].cpu().numpy()
        lengths = batch["lengths"].cpu().numpy()
        run_ids = batch["run_id"].cpu().numpy()
        for i in range(st.shape[0]):
            L = int(lengths[i])
            st_i = st[i, :L]
            ex_i = torch.tensor(ex[i, :L], dtype=torch.float32, device=device).unsqueeze(0)
            des_i = torch.tensor(des[i], dtype=torch.float32, device=device).unsqueeze(0)
            if mode == "teacher":
                pdelta = model.forward_teacher_forcing(
                    torch.tensor(st_i, device=device).unsqueeze(0),
                    ex_i,
                    des_i,
                ).cpu().numpy()[0]
                pred = np.zeros_like(st_i)
                pred[0] = st_i[0]
                pred[1:] = st_i[:-1] + pdelta[: L - 1]
            else:
                pred = model.forward_rollout(
                    torch.tensor(st_i[0], device=device).unsqueeze(0),
                    ex_i,
                    des_i,
                    lengths=torch.tensor([L], device=device),
                    max_delta_scaled=MAX_DELTA_SCALED,
                    max_state_scaled=MAX_STATE_SCALED,
                ).cpu().numpy()[0]
            true_phys = scaler_state.inverse_transform(st_i)
            pred_phys = scaler_state.inverse_transform(pred)
            if APPLY_PHYSICAL_CLAMP_EVAL:
                pred_phys = clamp_state_physical(pred_phys, state_cols, species_cols)
            rows.append({
                "run_id": int(run_ids[i]),
                "true": true_phys,
                "pred": pred_phys,
            })
    return rows


def anchor_rollout_rows(rows):
    """Pin inlet BC — same API/order as Main_7 §9."""
    if not rows:
        return rows
    all_true = np.concatenate([r["true"] for r in rows], axis=0)
    all_pred = np.concatenate([r["pred"] for r in rows], axis=0)
    run_ids_out = np.concatenate(
        [np.full(r["true"].shape[0], r["run_id"]) for r in rows]
    )
    rel_pos_out = np.concatenate([
        df.loc[full_run_id == r["run_id"]].sort_values("z_position_m")[
            "relative_position"
        ].to_numpy()
        for r in rows
    ])
    all_pred, n_anchored = anchor_inlet_profile_predictions(
        all_pred,
        all_true,
        run_ids_out,
        rel_pos_out,
    )
    print(f"Inlet anchoring (rollout): {n_anchored} row(s) set to Cantera truth.")
    offset = 0
    for r in rows:
        n = r["true"].shape[0]
        r["pred"] = all_pred[offset : offset + n]
        offset += n
    return rows


def stack_phys(rows):
    y_true = np.concatenate([r["true"] for r in rows], axis=0)
    y_pred = np.concatenate([r["pred"] for r in rows], axis=0)
    return y_true, y_pred


test_tf = collect_predictions(test_loader, mode="teacher")
test_ro = anchor_rollout_rows(collect_predictions(test_loader, mode="rollout"))
yt_ro, yp_ro = stack_phys(test_ro)

print(metrics_disclaimer(SMOKE_MODE))
print("Per-variable metrics (physical units; no pooled cross-unit R²):")
print_per_variable_metrics(test_tf, test_ro, state_cols, species_cols)

_env_ok = envelope_sanity_summary(test_ro, state_cols, species_cols, rel_margin=0.25)
print("\nRollout envelope sanity (pred within 25% margin of true min/max):")
for col, ok in _env_ok.items():
    print(f"  {col}: {'OK' if ok else 'FAIL'}")

for col in ["temperature_K", "pressure_Pa", "density_kgm3", "velocity_ms"]:
    j = state_cols.index(col)
    print(f"  Positivity violations {col}: {int((yp_ro[:, j] < 0).sum())}")

if species_cols:
    sp_idx = [state_cols.index(c) for c in species_cols]
    print(f"  Species |sum Y - 1| mean: {np.mean(np.abs(yp_ro[:, sp_idx].sum(1) - 1)):.4e}")

_m7 = EXPORT_DIR / "simple_nn_full_profile_manifest.json"
if _m7.exists():
    with open(_m7) as f:
        m7 = json.load(f)
    m7m = m7.get("metrics", m7)
    print(
        "Main_7 (MLP, same test_size/random_state, row-wise MLP): "
        f"test_r2={m7m.get('test_r2_uniform_avg', m7m.get('test_r2', 'n/a'))}"
    )
    if m7.get("data_file") and Path(m7["data_file"]).name != Path(DATA_FILE).name:
        print(f"[WARN] Main_7 used different pkl: {m7.get('data_file')}")
    print(
        "[NOTE] Main_8 holds out an extra val run fraction from train; "
        "test *runs* match Main_7 when data_file, test_size, random_state match."
    )
elif not SMOKE_MODE:
    print("[INFO] No Main_7 manifest - run Main_7 for comparison.")
elif SMOKE_MODE:
    print("[SMOKE] Skipping Main_7 metric comparison (development dataset).")

# Rollout diagnostics: where profiles leave the Cantera envelope
rel_pos_test = np.concatenate([
    df.loc[full_run_id == r["run_id"]].sort_values("z_position_m")["relative_position"].to_numpy()
    for r in test_ro
])
print("\nRollout min/max table (test, after inlet anchor):")
print(rollout_minmax_table(test_ro, state_cols, label="rollout").to_string(index=False))

_div = first_divergence_report(test_ro, state_cols, rel_pos_test, rel_threshold=0.25)
_div_np = first_nonphysical_divergence_report(
    test_ro, state_cols, species_cols, rel_pos_test, rel_threshold=0.25,
)
print(rollout_divergence_summary(_div))
print("\nFirst nonphysical / rel-err divergence (per run, variable):")
print(_div_np[_div_np["first_index"] >= 0].head(20).to_string(index=False))
print("\nFirst divergence (rel err > 25% vs Cantera), per run and variable:")
print(_div.to_string(index=False))


In [ ]:
# 9. PROFILE PLOTS (rollout temperature)
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

_n_plot = min(3, len(test_ro))
_t_ylim = plot_y_limits_from_true(test_ro, state_cols).get("temperature_K")
fig, axes = plt.subplots(_n_plot, 2, figsize=(12, 3.2 * _n_plot), squeeze=False)
t_idx = state_cols.index("temperature_K")
for i, r in enumerate(test_ro[:_n_plot]):
    g = df.loc[full_run_id == r["run_id"]].sort_values("z_position_m")
    x_rel = g["relative_position"].to_numpy()
    x_tau = g["relative_tau"].to_numpy()
    axes[i, 0].plot(x_rel, r["true"][:, t_idx], "k-", label="Cantera")
    axes[i, 0].plot(x_rel, r["pred"][:, t_idx], "r--", label="GRU rollout")
    axes[i, 0].set_xlabel("relative_position")
    axes[i, 0].set_ylabel("T [K]")
    axes[i, 0].legend(fontsize=8)
    axes[i, 1].plot(x_tau, r["true"][:, t_idx], "k-")
    axes[i, 1].plot(x_tau, r["pred"][:, t_idx], "r--")
    axes[i, 1].set_xlabel("relative_tau")
if IF_PLOT_EXPORT:
    fig.savefig(FIG_DIR / "gru_rollout_temperature_profiles.png", dpi=150, bbox_inches="tight")
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)


In [ ]:
# 10. EXPORT
import joblib

metrics = {
    "smoke_mode": SMOKE_MODE,
    "n_runs_total": N_RUNS_TOTAL,
    "test_teacher_forced_rmse": _rmse_tf,
    "test_rollout_rmse": _rmse_ro,
    "test_outlet_rollout_rmse": _out_rmse,
    "best_val_rollout_mse_scaled": best_val,
    "early_stopping_metric": EARLY_STOPPING_METRIC,
}
if not SMOKE_MODE:
    metrics["test_teacher_forced_r2"] = _r2_tf
    metrics["test_rollout_r2"] = _r2_ro

if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stem = "gru_profile_evolution"
    torch.save(model.state_dict(), EXPORT_DIR / f"{stem}.pt")
    joblib.dump(
        {
            "scaler_design": scaler_design,
            "scaler_state": scaler_state,
            "scaler_exog": scaler_exog,
            "label_encoder": label_encoder,
        },
        EXPORT_DIR / f"{stem}_scalers.joblib",
    )
    manifest = {
        "model": "ProfileGRU",
        "notebook": MAIN_8_STEM,
        "data_file": Path(DATA_FILE).name,
        "design_cols": spec.design_cols,
        "exog_cols": spec.exog_cols,
        "state_cols": state_cols,
        "species_cols": species_cols,
        "run_cols": run_cols,
        "hidden_size": HIDDEN_SIZE,
        "design_hidden": DESIGN_HIDDEN,
        "num_gru_layers": NUM_GRU_LAYERS,
        "dropout": DROPOUT,
        "smoke_mode": SMOKE_MODE,
        "n_runs_total": N_RUNS_TOTAL,
        "metrics": metrics,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "exported": datetime.now().isoformat(timespec="seconds"),
    }
    with open(EXPORT_DIR / f"{stem}_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Exported {stem}.pt, scalers, manifest")
print(json.dumps(metrics, indent=2))
